In [1]:
import numpy as np

from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from cart import BinaryDecisionTree
from utils import cat_label_convert

from typing import Tuple, List, Any, Dict


这里依然是 [[Sigmoid 函数]]。

$$\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$$

In [2]:
def sigmoid(x: np.ndarray) -> np.ndarray:
    """定义 Sigmoid 激活函数

    Args:
        x (np.ndarray): 输入的数组或标量

    Returns:
        np.ndarray: Sigmoid 转换后的结果
    """
    # 实现 Sigmoid 函数公式
    return 1 / (1 + np.exp(-x))

在 XGBoost 二分类任务中，模型（决策树）直接输出的值并非概率，而是**原始得分 $z$**（代码中的 `y_pred`）。但交叉熵损失函数需要输入**概率 $p$**。因此，在求导时不仅要对交叉熵求导，还要结合激活函数使用链式法则，求出损失函数对原始得分 $z$ 的导数。

**1. 定义前向传播公式**

首先，将原始得分 $z$ 通过 Sigmoid 激活函数转化为预测概率 $p$：

$$p = \sigma(z) = \frac{1}{1 + e^{-z}}$$

然后，计算二元交叉熵损失函数 $L$：

$$L = - \left[ y \cdot \ln(p) + (1-y) \cdot \ln(1 - p) \right]$$

**2. 计算一阶导数（Gradient, $g$）**

我们需要求损失函数 $L$ 对原始得分 $z$ 的一阶导数 $\frac{\partial L}{\partial z}$。根据链式法则：

$$\frac{\partial L}{\partial z} = \frac{\partial L}{\partial p} \cdot \frac{\partial p}{\partial z}$$

- **链式前半部分**（损失函数对概率 $p$ 的导数）：

    $$\frac{\partial L}{\partial p} = -\frac{y}{p} + \frac{1-y}{1-p} = \frac{p - y}{p(1-p)}$$

- **链式后半部分**（Sigmoid 函数对原始得分 $z$ 的导数）：

    $$\frac{\partial p}{\partial z} = p(1-p)$$

- **合并链式法则**：

    $$g = \frac{\partial L}{\partial z} = \frac{p - y}{p(1-p)} \times p(1-p) = p - y$$


In [3]:
def logistic_loss_gradient(y: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """计算对数损失函数（Logit Loss）的一阶梯度

    Args:
        y (np.ndarray): 真实标签
        y_pred (np.ndarray): 模型的原始预测值

    Returns:
        np.ndarray: 对应的一阶梯度数组
    """
    # 使用 sigmoid 函数并计算一阶梯度 (参考书中 g_i 的推导思想)
    p = sigmoid(y_pred)
    return p - y

这里需要求出损失函数 $L$ 对原始得分 $z$ 的二阶导数 $\frac{\partial^2 L}{\partial z^2}$。

既然一阶导数为 $g = p - y$，那么二阶导数 $h$ 就是对一阶导数再求一次导：

$$h = \frac{\partial g}{\partial z} = \frac{\partial (p - y)}{\partial z}$$

根据求导的加减法则，拆开式子：

$$h = \frac{\partial p}{\partial z} - \frac{\partial y}{\partial z}$$

已知真实标签 $y$ 对于变量 $z$ 而言是一个固定常数，因此对常数 $y$ 求导结果为 $0$。同时结合前面推导的 Sigmoid 导数：

$$\frac{\partial p}{\partial z} = p(1 - p)$$

则最终二阶导数的结果为：

$$h = p(1 - p)$$

In [4]:
def logistic_loss_hessian(y: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """计算对数损失函数（Logit Loss）的二阶梯度（黑塞矩阵）

    Args:
        y (np.ndarray): 真实标签
        y_pred (np.ndarray): 模型的原始预测值

    Returns:
        np.ndarray: 对应的二阶梯度数组
    """
    # 使用 sigmoid 函数并计算二阶梯度 (参考书中 h_i 的推导思想)
    p = sigmoid(y_pred)
    return p * (1 - p)

这里我们需要实现这个公式的其中一个节点：

$$Gain = \frac{1}{2} \left[ \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L + G_R)^2}{H_L + H_R + \lambda} \right] - \gamma$$

即

* 左子结点得分：$\frac{G_L^2}{H_L + \lambda}$
* 右子结点得分：$\frac{G_R^2}{H_R + \lambda}$
* 父结点得分：$\frac{(G_L + G_R)^2}{H_L + H_R + \lambda}$

中的任意一项。

其中 $G$ 和 $H$ 的求解方法已经在上面定义了，调用即可。

In [5]:
def node_gain(y_true: np.ndarray, y_pred: np.ndarray, lambda_reg: float = 0.0) -> float:
    """计算单个结点的基础信息增益得分
    注意：为了更严谨，这里在函数签名中加入了 lambda_reg 参数。
    书中代码清单 12-1 未显示定义 lambda，默认将其视为 0。

    Args:
        y_true (np.ndarray): 当前结点内的真实标签
        y_pred (np.ndarray): 当前结点内的预测值
        lambda_reg (float): L2 正则化系数

    Returns:
        float: 结点的增益得分 (类似 G^2 / (H + lambda) 的部分)
    """
    # 1. 计算当前结点所有样本的一阶梯度求和 (G)
    G = np.sum(logistic_loss_gradient(y_true, y_pred))
    # 2. 计算当前结点所有样本的二阶梯度求和 (H)
    H = np.sum(logistic_loss_hessian(y_true, y_pred))
    # 3. 返回该结点的增益得分公式结果
    return (G ** 2) / (H + lambda_reg)

这里正式计算 gain 值

$$Gain = \frac{1}{2} \left[ \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L + G_R)^2}{H_L + H_R + \lambda} \right] - \gamma$$

所有的节点值均可以通过上面 `node_gain()` 得出

In [6]:
def calculate_split_gain(y_left: np.ndarray,
                         y_pred_left: np.ndarray,
                         y_right: np.ndarray,
                         y_pred_right: np.ndarray,
                         lambda_reg: float = 1.0,
                         gamma: float = 0.0) -> float:
    """计算某次特定划分方案的完整信息增益 (包含 gamma 惩罚)

    Args:
        y_left, y_pred_left: 左子集数据
        y_right, y_pred_right: 右子集数据
        lambda_reg: L2 正则化系数
        gamma: 树复杂度（叶子结点数）惩罚项

    Returns:
        float: 完整的 Gain 值
    """
    # 1. 计算左子集的 gain 节点
    left_node_gain = node_gain(y_left, y_pred_left, lambda_reg)
    # 2. 计算右子集的 gain 节点
    right_node_gain = node_gain(y_right, y_pred_right, lambda_reg)
    # 3. 计算总叶子集的 gain 节点
    father_node_gain = node_gain(np.hstack((y_left, y_right)), np.hstack((y_pred_left, y_pred_right)), lambda_reg)
    # 4. 代入我们在笔记中总结的完整 Gain 公式：
    gain = 0.5 * (left_node_gain + right_node_gain - father_node_gain) - gamma
    # 5. 返回计算结果
    return gain

叶子的权重：

$$w_j^* = -\frac{G_j}{H_j + \lambda}$$

In [7]:
def calculate_leaf_weight(y_true: np.ndarray, y_pred: np.ndarray, lambda_reg: float = 0.0) -> float:
    """计算叶子结点的最优输出权重

    Args:
        y_true (np.ndarray): 落在该叶子结点的真实标签
        y_pred (np.ndarray): 落在该叶子结点的预测值
        lambda_reg (float): L2 正则化系数

    Returns:
        float: 最优叶子结点权重 w*
    """
    # 1. 计算当前结点所有样本的一阶梯度求和 (G)
    G = np.sum(logistic_loss_gradient(y_true, y_pred))
    # 2. 计算当前结点所有样本的二阶梯度求和 (H)
    H = np.sum(logistic_loss_hessian(y_true, y_pred))
    # 3. 返回带有正则化的最优权重计算结果
    return - G / (H + lambda_reg)

In [8]:
def split_dataset(X: np.ndarray,
                  y: np.ndarray,
                  y_pred: np.ndarray,
                  feature_idx: int,
                  threshold: float) \
                -> Dict[str, np.ndarray]:
    """根据指定的特征和阈值，将数据集严格划分为左右两个子集

    Args:
        X, y, y_pred: 当前结点的数据
        feature_idx: 用于切分的特征索引
        threshold: 切分阈值

    Returns:
        Dict: 依次包含 (X_left, X_right, y_left, y_right, y_pred_left, y_pred_right)
    """
    # 1. 找到 X[:, feature_idx] <= threshold 的布尔索引
    mask = X[:, feature_idx] <= threshold
    # 2. 利用布尔索引将 X, y, y_pred 分别切片出 left 和 right 部分
    X_left = X[mask]
    X_right = X[~mask]
    y_left = y[mask]
    y_right = y[~mask]
    y_pred_left = y_pred[mask]
    y_pred_right = y_pred[~mask]
    # 3. 返回这 6 个切分好的字典
    return {
        'X_left': X_left,
        'X_right': X_right,
        'y_left': y_left,
        'y_right': y_right,
        'y_pred_left': y_pred_left,
        'y_pred_right': y_pred_right
    }


In [9]:
def find_best_split_for_feature(X: np.ndarray, y: np.ndarray, y_pred: np.ndarray, feature_idx: int, lambda_reg: float = 1.0, gamma: float = 0.0) -> Dict[str, Any]:
    """在单个指定的特征上，寻找能带来最大增益的切分阈值

    Returns:
        Dict: 包含该特征下最大增益 'max_gain' 和最佳阈值 'best_threshold'
    """
    # 1. 获取该特征列 X[:, feature_idx] 的所有唯一值 (np.unique) 作为候选阈值
    thresholds = np.unique(X[:, feature_idx])
    # 2. 初始化最佳阈值和 max_gain，max_gain 为一个极小值 (或 0)
    max_gain = 0
    best_threshold = None
    # 3. 遍历候选阈值：
    for threshold in thresholds:
        # a. 调用 split_dataset 划分数据
        split_data = split_dataset(X, y, y_pred, feature_idx, threshold)
        # b. 调用 calculate_split_gain 计算增益
        gain = calculate_split_gain(split_data['y_left'],
                                    split_data['y_pred_left'],
                                    split_data['y_right'],
                                    split_data['y_pred_right'],
                                    lambda_reg, gamma)
        # c. 如果增益更大，则更新 max_gain 和 best_threshold
        if gain > max_gain:
            max_gain = gain
            best_threshold = threshold
    # 4. 返回记录了该特征最佳表现的字典

    return {
        'max_gain': max_gain,
        'threshold': best_threshold,
    }


In [10]:
def find_best_split(X: np.ndarray, y: np.ndarray, y_pred: np.ndarray, lambda_reg: float = 1.0, gamma: float = 0.0) -> Dict[str, Any]:
    """遍历所有特征，寻找全局最佳的分裂方案 (组合并调用前面的函数)

    Returns:
        Dict: 包含全局最佳特征索引 'feature_idx'、阈值 'threshold'、以及切分后的左右数据
    """
    best_gain = 0
    best_feature_idx = None
    best_threshold = None
    # 1. 遍历所有特征列的索引
    for i in range(X.shape[1]):
        # 2. 对每个特征调用 find_best_split_for_feature 得到其最佳阈值和增益
        current_feature_best_split = find_best_split_for_feature(X, y, y_pred, i, lambda_reg, gamma)
        # 3. 比较所有特征的增益，找出全局最大的 Gain
        if current_feature_best_split['max_gain'] > best_gain:
            best_gain = current_feature_best_split['max_gain']
            best_feature_idx = i
            best_threshold = current_feature_best_split['threshold']
    # 4. 如果最大 Gain > 0，利用最佳特征和阈值调用 split_dataset 获取切分后的数据
    if best_gain > 0:
        split_data = split_dataset(X, y, y_pred, best_feature_idx, best_threshold)
        split_data['feature_idx'] = best_feature_idx
        split_data['threshold'] = best_threshold
        return split_data
    # 5. 返回包含完整分裂信息的字典（如果没有正增益，返回空字典表示停止分裂）
    else:
        return None

In [11]:
def is_terminal_node(current_depth: int, max_depth: int, n_samples: int, min_samples_split: int = 2) -> bool:
    """检查当前结点是否满足停止生长的条件（预剪枝逻辑）

    Returns:
        bool: 如果满足停止条件返回 True，否则返回 False
    """
    # 1. 如果 current_depth >= max_depth，返回 True
    # 2. 如果 n_samples < min_samples_split，返回 True
    if current_depth >= max_depth or n_samples < min_samples_split:
        return True

    # 3. 否则返回 False
    else:
        return False

这里是构建单颗树的函数，这棵树主要保存以下信息作为单颗树的预测：

1. 终点，即叶子节点：
- 是否为叶子：标志位（`is_leaf: True`）。
- 叶子权重：用 $\frac{G}{H+\lambda}$​ 算出来的权重数值 $w^*$ 。

2. 岔路，即内部节点：
- 是否为叶子：和叶子节点一样的标志位（`is_leaf: False`）。
- 分流规则：根据遍历得到的特征：`feature_idx` 和阈值： `threshold`，该岔路的关键分支信息。
- 左右子树：它会包含 `left` 和 `right` 两个键，这两个键对应的值就是下一个分支的信息，即又开始判断是终点还是岔路。

In [12]:
def build_tree(X: np.ndarray,
               y: np.ndarray,
               y_pred: np.ndarray,
               current_depth: int = 0,
               max_depth: int = 3,
               min_samples_split: int = 2,
               lambda_reg: float = 1,
               gamma: float = 0.0) -> Dict[str, Any]:
    """组装引擎：递归构建 XGBoost 决策树
    """
    # 1. 调用 is_terminal_node 检查是否停止。如果是，调用之前写的 calculate_leaf_weight 计算权重并返回叶子字典。
    if is_terminal_node(current_depth, max_depth, len(X), min_samples_split):
        return {
            "is_leaf": True,
            "weight": calculate_leaf_weight(y, y_pred, lambda_reg),
        }
    # 2. 调用 find_best_split 尝试寻找全局最优切分。
    best_split = find_best_split(X, y, y_pred, lambda_reg, gamma)
    # 3. 如果 find_best_split 返回空（代表增益为负或无法切分），同样计算叶子权重并返回。
    if best_split is None:
        return {
            "is_leaf": True,
            "weight": calculate_leaf_weight(y, y_pred, lambda_reg),
        }
    # 4. 找到有效切分，提取出切分好的 left 和 right 数据。递归调用：
    left_node = build_tree(best_split['X_left'], best_split['y_left'], best_split['y_pred_left'], current_depth + 1, max_depth, min_samples_split, lambda_reg, gamma)
    right_node = build_tree(best_split['X_right'], best_split['y_right'], best_split['y_pred_right'], current_depth + 1, max_depth, min_samples_split, lambda_reg, gamma)
    # 5. 返回包含特征索引、阈值以及 left_node 和 right_node 的内部结点字典。
    return {
        "is_leaf": False,
        "feature_idx": best_split["feature_idx"],
        "threshold": best_split["threshold"],
        "left_node": left_node,
        "right_node": right_node,
    }


In [13]:
def predict_single_sample(tree: Dict[str, Any], x: np.ndarray) -> float:
    """使用训练好的单棵决策树，对单个样本进行预测

    Args:
        tree (Dict): build_tree 返回的嵌套字典
        x (np.ndarray): 单个样本的特征向量 (1 维数组)

    Returns:
        float: 该样本落入的叶子结点的权重 (weight)
    """
    # 1. 检查当前结点 tree["is_leaf"] 是否为 True。
    if tree['is_leaf']:
        # 如果是，直接返回 tree["weight"]。
        return tree['weight']
    # 2. 判断该样本的对应特征值 x[feature_idx] 是否 <= threshold。
    if x[tree["feature_idx"]] <= tree["threshold"]:
        # 4. 如果是，递归调用 predict_single_sample，传入 tree["left_node"] 和 x。
        weight = predict_single_sample(tree['left_node'], x)
        # 5. 如果不是，递归调用 predict_single_sample，传入 tree["right_node"] 和 x。
    else:
        weight = predict_single_sample(tree['right_node'], x)

    return weight

模型最后返回的结构如下：
```mermaid
---
config:
  theme: redux-dark-color
---
mindmap
((模型数据结构))
  全局参数
    learning_rate<br> 学习率。预测时，每棵树的输出权重乘以它，用于控制更新步伐<br> 必须 | Float
    n_estimators<br>树的总数。通过读取树列表的长度自动得知<br>可选 | Integer
    base_score<br>初始预测值。预测时所有的得分都要在它的基础上进行累加<br>必须 | Float
  树
    内部结点
      feature_idx<br>决定当前岔路根据哪一列特征进行判断。<br>必须 | Integer
      threshold<br>决定当前路口的阈值，例如：当前特征是否小于等于 2.45。<br> 必须 | Float
      left and right<br>指向下一个树的子结点指针，嵌套的字典结构<br>必须 | Dict
    叶子结点
      weight<br>该结点的最终预测输出得分，即叶子权重 w^*<br>必须 | Float
      is_leaf<br>标志位，直接获取可以判断是否是叶子节点<br>必须 | Boolean
```

In [14]:
def fit_xgboost(X: np.ndarray, y: np.ndarray, n_estimators: int = 30, learning_rate: float = 0.001) -> List[Any]:
    """训练 XGBoost 分类模型，通过加法模型逐步拟合多棵树

    Args:
        X (np.ndarray): 训练集特征数据
        y (np.ndarray): 训练集标签数据
        n_estimators (int): 迭代次数，即要训练的树的棵数
        learning_rate (float): 学习率（收缩步长）

    Returns:
        List[Any]: 训练好的决策树列表（加性模型）
    """
    # 初始化一个空列表用来保存训练好的树
    trees = []

    # 初始化一个全为 0 的数组，作为最初始的预测值 y_pred
    y_pred = np.zeros(y.shape)
    base_score = np.mean(y_pred)
    # 开始迭代训练每一棵树
    for i in range(n_estimators):
        # 3. 实例化一棵单棵 XGBoost 树并进行拟合 (fit)
        tree = build_tree(X, y, y_pred)
        # 4. 用刚训练好的这棵树对 X 进行预测，得到 iter_pred
        iter_pred = np.array([predict_single_sample(tree, x) for x in X])
        # 5. 将这棵树的预测结果 iter_pred 乘以 learning_rate 后，累加到总的 y_pred 中
        y_pred = y_pred + iter_pred * learning_rate
        # 6. 将训练好的树追加到 trees 列表中
        trees.append(tree)
    return {
        'learning_rate': learning_rate,
        'n_estimators': n_estimators,
        'base_score': base_score,
        'trees': trees,
    }

In [15]:
def predict_xgboost(X: np.ndarray, trees: List[Any], base_score, learning_rate: float = 0.001) -> np.ndarray:
    """使用训练好的 XGBoost 分类模型进行预测

    Args:
        X (np.ndarray): 测试集特征数据
        trees (List[Any]): fit 函数返回的训练好的树列表
        learning_rate (float): 学习率，需与训练时保持一致

    Returns:
        np.ndarray: 最终的类别预测结果
    """
    # 1. 初始化一个全 0 矩阵用于累加预测值
    y_perd = np.full(X.shape[0], base_score)
    # 2. 遍历 trees 列表中的每一棵树，让它们分别对 X 进行预测\
    for tree in trees:
        iter_pred = np.array([predict_single_sample(tree, x) for x in X])
        # 3. 将每棵树的预测结果乘以 learning_rate 后累加起来
        y_perd = y_perd + iter_pred * learning_rate
    # 4. 对累加后的原始得分进行转换，得到最终的类别标签
    y_perd = sigmoid(y_perd)
    return (y_perd >= 0.5).astype(int)

In [16]:
# 导入鸢尾花数据集
data = datasets.load_breast_cancer()
# 获取输入输出
X, y = data.data, data.target
# 数据集划分
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=43)

In [17]:
print(f'X_train: {X_train.shape}\n' \
      f'y_train: {y_train.shape}\n' \
      f'X_test: {X_test.shape}\n' \
      f'y_test: {y_test.shape}')

X_train: (455, 30)
y_train: (455,)
X_test: (114, 30)
y_test: (114,)


In [18]:
# 模型拟合
model = fit_xgboost(X_train, y_train)

In [19]:
# 模型预测
y_pred = predict_xgboost(X_test, model['trees'], model['base_score'], model['learning_rate'])
# 准确率评估
accuracy = accuracy_score(y_test, y_pred)
print ("Accuracy: ", accuracy)

Accuracy:  0.956140350877193
